In [1]:
import yaml
import json
import logging
import joblib
from datetime import datetime
from pathlib import Path
from sklearn.base import clone

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

CONFIGS_DIR = Path('../configs')
CONFIGS_DIR.mkdir(parents=True, exist_ok=True)

ARTIFACTS_ROOT = Path('../artifacts')
METRICS_DIR = ARTIFACTS_ROOT / 'metrics'
MODELS_DIR = ARTIFACTS_ROOT / 'models'
FIGURES_EXP = ARTIFACTS_ROOT / 'figures' / '01_exp'
CATBOOST_INFO_DIR = ARTIFACTS_ROOT / 'catboost_info'

for p in [METRICS_DIR, MODELS_DIR, FIGURES_EXP, CATBOOST_INFO_DIR]:
    p.mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def save_figure(fig, name):
    filepath = FIGURES_EXP / f'{name}.png'
    fig.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.close(fig)
    return filepath

# --- Сохранение конфигурации эксперимента ---
experiment_config = {
    'random_state': 42,
    'test_size': 0.2,
    'preprocessing': {
        'numeric_scaler': 'StandardScaler',
        'categorical_encoder': 'OneHotEncoder',
        'handle_unknown': 'ignore'
    },
    'models': {
        'LinearRegression': {},
        'RandomForest': {'n_estimators': 150, 'n_jobs': -1},
        'CatBoost': {'iterations': 500, 'learning_rate': 0.05, 'verbose': 0},
        'Stacking': {
            'cv': 5,
            'n_jobs': -1,
            'base_models': {
                'lr': {},
                'rf': {'n_estimators': 100, 'n_jobs': -1},
                'cb': {'iterations': 300, 'verbose': 0}
            },
            'final_estimator': 'LinearRegression'
        }
    }
}
with open(CONFIGS_DIR / 'experiment.yaml', 'w', encoding='utf-8') as f:
    yaml.safe_dump(experiment_config, f, default_flow_style=False, allow_unicode=True)
print(f'Конфигурация эксперимента сохранена: {CONFIGS_DIR / "experiment.yaml"}')

# --- Сохранение конфигурации сервиса ---
serving_config = {
    'artifacts_dir': '../artifacts',
    'risk_thresholds': {'low': 20.0, 'medium': 35.0},
    'server': {'host': '0.0.0.0', 'port': 8000}
}
with open(CONFIGS_DIR / 'serving.yaml', 'w', encoding='utf-8') as f:
    yaml.safe_dump(serving_config, f, default_flow_style=False, allow_unicode=True)
print(f'Конфигурация сервиса сохранена: {CONFIGS_DIR / "serving.yaml"}')

# Загрузка конфигов для использования
with open(CONFIGS_DIR / 'data.yaml', 'r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)
with open(CONFIGS_DIR / 'experiment.yaml', 'r', encoding='utf-8') as f:
    exp_cfg = yaml.safe_load(f)
with open(CONFIGS_DIR / 'serving.yaml', 'r', encoding='utf-8') as f:
    serve_cfg = yaml.safe_load(f)

Конфигурация эксперимента сохранена: ..\configs\experiment.yaml
Конфигурация сервиса сохранена: ..\configs\serving.yaml


In [2]:
df = pd.read_csv(
    data_cfg['source_path'],
    sep=data_cfg['separator'],
    decimal=data_cfg['decimal']
)
df = df.dropna(how=data_cfg['dropna_how']).reset_index(drop=True)

num_cols = data_cfg['columns']['numeric']
cat_cols = data_cfg['columns']['categorical']
target_cols = data_cfg['columns']['targets']

df[cat_cols] = df[cat_cols].astype('category')

X = df.drop(columns=target_cols)
y = df[target_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=exp_cfg['test_size'], random_state=exp_cfg['random_state']
)

logger.info(f'Data loaded: train={len(X_train)}, test={len(X_test)}')

2026-05-17 21:38:23,976 - INFO - Data loaded: train=614, test=154


In [3]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown=exp_cfg['preprocessing']['handle_unknown'], sparse_output=False), cat_cols)
    ]
)

models_cfg = exp_cfg['models']
models = {
    'LinearRegression': Pipeline([
        ('prep', preprocessor),
        ('model', LinearRegression())
    ]),
    'RandomForest': Pipeline([
        ('prep', preprocessor),
        ('model', RandomForestRegressor(
            n_estimators=models_cfg['RandomForest']['n_estimators'],
            random_state=exp_cfg['random_state'],
            n_jobs=models_cfg['RandomForest']['n_jobs']
        ))
    ]),
    'CatBoost': Pipeline([
        ('prep', preprocessor),
        ('model', CatBoostRegressor(
            verbose=models_cfg['CatBoost']['verbose'],
            random_state=exp_cfg['random_state'],
            iterations=models_cfg['CatBoost']['iterations'],
            learning_rate=models_cfg['CatBoost']['learning_rate'],
            loss_function='RMSE'
        ))
    ]),
    'Stacking': Pipeline([
        ('prep', preprocessor),
        ('model', StackingRegressor(
            estimators=[
                ('lr', LinearRegression()),
                ('rf', RandomForestRegressor(
                    n_estimators=models_cfg['Stacking']['base_models']['rf']['n_estimators'],
                    random_state=exp_cfg['random_state'],
                    n_jobs=models_cfg['Stacking']['base_models']['rf']['n_jobs']
                )),
                ('cb', CatBoostRegressor(
                    verbose=models_cfg['Stacking']['base_models']['cb']['verbose'],
                    random_state=exp_cfg['random_state'],
                    iterations=models_cfg['Stacking']['base_models']['cb']['iterations']
                ))
            ],
            final_estimator=LinearRegression(),
            cv=models_cfg['Stacking']['cv'],
            n_jobs=models_cfg['Stacking']['n_jobs']
        ))
    ])
}

logger.info('Model pipelines created')

2026-05-17 21:38:23,997 - INFO - Model pipelines created


In [4]:
results = []
fitted_pipelines = {}

for target in target_cols:
    logger.info(f'Training for target: {target}')
    y_tr = y_train[target]
    y_te = y_test[target]
    
    for name, pipe_template in models.items():
        pipe = clone(pipe_template)
        logger.info(f'  Training {name}...')
        
        if name == 'CatBoost':
            pipe.set_params(model__train_dir=str(CATBOOST_INFO_DIR / f'cb_{target}'))
        
        pipe.fit(X_train, y_tr)
        y_pred_train = pipe.predict(X_train)
        y_pred_test = pipe.predict(X_test)
        
        metrics_train = {
            'mae': mean_absolute_error(y_tr, y_pred_train),
            'rmse': np.sqrt(mean_squared_error(y_tr, y_pred_train)),
            'r2': r2_score(y_tr, y_pred_train)
        }
        metrics_test = {
            'mae': mean_absolute_error(y_te, y_pred_test),
            'rmse': np.sqrt(mean_squared_error(y_te, y_pred_test)),
            'r2': r2_score(y_te, y_pred_test)
        }
        
        results.append({
            'Target': target, 'Model': name,
            'MAE_train': metrics_train['mae'], 'RMSE_train': metrics_train['rmse'], 'R2_train': metrics_train['r2'],
            'MAE_test': metrics_test['mae'], 'RMSE_test': metrics_test['rmse'], 'R2_test': metrics_test['r2']
        })
        
        fitted_pipelines[f'{name}_{target}'] = pipe

results_df = pd.DataFrame(results)
logger.info('Training completed')

2026-05-17 21:38:24,021 - INFO - Training for target: Y1
2026-05-17 21:38:24,024 - INFO -   Training LinearRegression...
2026-05-17 21:38:24,094 - INFO -   Training RandomForest...
2026-05-17 21:38:24,573 - INFO -   Training CatBoost...
2026-05-17 21:38:25,699 - INFO -   Training Stacking...
2026-05-17 21:38:37,613 - INFO - Training for target: Y2
2026-05-17 21:38:37,614 - INFO -   Training LinearRegression...
2026-05-17 21:38:37,634 - INFO -   Training RandomForest...
2026-05-17 21:38:37,945 - INFO -   Training CatBoost...
2026-05-17 21:38:38,303 - INFO -   Training Stacking...
2026-05-17 21:38:40,140 - INFO - Training completed


In [5]:
results_df.to_csv(METRICS_DIR / 'results_summary.csv', index=False, float_format='%.4f')
logger.info(f'Saved results summary: {METRICS_DIR / "results_summary.csv"}')

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x='Model', y='R2_test', hue='Target', data=results_df, ax=ax)
ax.set_title('R2 на тестовой выборке')
ax.set_xlabel('Model')
ax.set_ylabel('R2')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
save_figure(fig, 'models_comparison_r2')
logger.info(f'Saved figure: {FIGURES_EXP / "models_comparison_r2.png"}')

2026-05-17 21:38:40,162 - INFO - Saved results summary: ..\artifacts\metrics\results_summary.csv
2026-05-17 21:38:40,512 - INFO - Saved figure: ..\artifacts\figures\01_exp\models_comparison_r2.png


In [6]:
avg_r2_test = results_df.groupby('Model')['R2_test'].mean().sort_values(ascending=False)
final_model = avg_r2_test.index[0]

selection_record = {
    'selected_model': final_model,
    'selection_criteria': 'max_mean_R2_test',
    'scores': {k: round(v, 4) for k, v in avg_r2_test.items()},
    'timestamp': datetime.now().isoformat(),
    'note': 'Model selected for deployment based on highest mean test R2'
}
with open(METRICS_DIR / 'model_selection.json', 'w', encoding='utf-8') as f:
    json.dump(selection_record, f, indent=2, ensure_ascii=False)
logger.info(f'Saved model selection: {METRICS_DIR / "model_selection.json"}')

for target in target_cols:
    key = f'{final_model}_{target}'
    model_path = MODELS_DIR / f'{key}.pkl'
    joblib.dump(fitted_pipelines[key], model_path)
    logger.info(f'Saved model artifact: {model_path}')

best_model_manifest = {
    'model_name': final_model,
    'targets': target_cols,
    'artifact_files': [f'{final_model}_{t}.pkl' for t in target_cols],
    'timestamp': datetime.now().isoformat(),
    'selection_criteria': selection_record['selection_criteria'],
    'preprocessor': 'ColumnTransformer(StandardScaler + OneHotEncoder)',
    'features': {'numeric': num_cols, 'categorical': cat_cols}
}
with open(MODELS_DIR / 'best_model_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(best_model_manifest, f, indent=2, ensure_ascii=False)
logger.info(f'Manifest saved: {MODELS_DIR / "best_model_manifest.json"}')

2026-05-17 21:38:40,540 - INFO - Saved model selection: ..\artifacts\metrics\model_selection.json


2026-05-17 21:38:40,592 - INFO - Saved model artifact: ..\artifacts\models\Stacking_Y1.pkl
2026-05-17 21:38:40,627 - INFO - Saved model artifact: ..\artifacts\models\Stacking_Y2.pkl
2026-05-17 21:38:40,630 - INFO - Manifest saved: ..\artifacts\models\best_model_manifest.json


In [7]:
print('Проверка воспроизводимости:')
print(f'Размер данных: {df.shape}')
print(f'Пропуски: {df.isna().sum().sum()}')
print(f'Дубликаты: {df.duplicated().sum()}')
for target in target_cols:
    print(f'{target}: диапазон [{df[target].min():.2f}, {df[target].max():.2f}]')
print(f'Финальная модель: {final_model}')
print(f'Конфиги сохранены в: {CONFIGS_DIR}')
print(f'Артефакты в metrics/: {[f.name for f in METRICS_DIR.iterdir()]}')
print(f'Артефакты в models/: {[f.name for f in MODELS_DIR.iterdir()]}')
print('Эксперимент завершён.')

Проверка воспроизводимости:
Размер данных: (768, 10)
Пропуски: 0
Дубликаты: 0
Y1: диапазон [6.01, 43.10]
Y2: диапазон [10.90, 48.03]
Финальная модель: Stacking
Конфиги сохранены в: ..\configs
Артефакты в metrics/: ['model_selection.json', 'results_summary.csv']
Артефакты в models/: ['best_model_manifest.json', 'Stacking_Y1.pkl', 'Stacking_Y2.pkl']
Эксперимент завершён.
